# Assignment 4 (new data) — 0. Inventory of `data/`

**Before any model is trained, establish what is actually on disk.**

The three datasets in `data/` arrive with their own `*_info.md` description files. Those files describe
what the datasets *are*, not necessarily what was *downloaded*. This notebook trusts neither: it walks the
directories, counts files, opens images and profiles the CSV, and reports the result.

That distinction is not pedantry. One of the three datasets turns out to be incomplete in a way that makes
supervised training on it impossible, and it is much cheaper to discover that here than three hours into a
training run.

| # | Dataset | Declared in `*_info.md` | Model family |
|---|---------|------------------------|--------------|
| **A** | CDC BRFSS diabetes 2015+2023 | 546,166 rows × 20 cols, 3 classes | **MLP** |
| **B** | Rice Image Dataset | 75,000 images, 5 classes | **CNN** |
| **C** | Intel Image Classification | ~25,000 images, 6 classes | **CNN** |

In [ ]:
import os, sys, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))        # notebook/  -> ass4_newdata
sys.path.insert(0, os.path.abspath(".."))       # repo root  -> ass4_utils, scratch_nn

import ass4_newdata as D
import ass4_utils as U

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 130)

print("repo root:", D.ROOT)
print("data dir :", D.DATA)

---
## 1. What is on disk

`D.inventory()` walks every declared directory and counts the image files it really contains. The
`usable_for_training` column is the one that matters: a split is only usable if it has **class folders**,
because the folder name *is* the label.

In [ ]:
inv = D.inventory()
display(inv)

print("\ntotal items on disk:", f"{inv['items'].sum():,}")
print("usable for supervised training:",
      f"{inv.loc[inv['usable_for_training'], 'items'].sum():,}")
print("present but unusable (no labels):",
      f"{inv.loc[inv['present'] & ~inv['usable_for_training'], 'items'].sum():,}")

> ### Finding: the Intel dataset is incomplete
>
> `data/` contains `seg_pred/` — 7,301 loose `.jpg` files with **no class sub-folders** — but neither
> `seg_train/` nor `seg_test/`. `seg_pred` is the original competition's *prediction* set: it never had
> public labels. Without `seg_train`/`seg_test` there is no ground truth, so on this dataset it is
> impossible to train a classifier, and equally impossible to measure an accuracy.
>
> Notebook `03_intel_scene_cnn.ipynb` is written and ready, and calls `D.require_intel_labels()` as its
> first action so it stops immediately with download instructions rather than failing halfway. `seg_pred`
> is still useful for one thing — an unlabelled inference demo — and the notebook uses it for exactly that.

---
## 2. Dataset A — BRFSS diabetes (tabular)

The claims in `data/diabetes_info.md` that the notebooks depend on are checked here directly: the row
count, the absence of missing values, the class split, and the 2015/2023 balance.

In [ ]:
df = pd.read_csv(D.DIABETES_CSV)

print(f"shape            {df.shape}")
print(f"missing values   {int(df.isna().sum().sum())}")
print(f"duplicate rows   {int(df.duplicated().sum()):,}  (kept on purpose - see info file)")
print(f"memory           {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

display(df.head())
display(df.describe().T[["min", "max", "mean", "std"]].round(2))

In [ ]:
# Target balance overall and per survey year - the imbalance drives how the
# results in notebook 01 have to be read.
counts = df[D.BRFSS_TARGET].value_counts().sort_index()
by_year = pd.crosstab(df["year"], df[D.BRFSS_TARGET], normalize="index")
by_year.columns = D.BRFSS_CLASSES

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bars = ax[0].bar(D.BRFSS_CLASSES, counts.values, color=["#4C78A8", "#F58518", "#E45756"])
ax[0].set_title("Target balance (Diabetes_012)")
ax[0].set_ylabel("respondents")
for b, c in zip(bars, counts.values):
    ax[0].text(b.get_x() + b.get_width() / 2, c, f"{c:,}\n{c / counts.sum():.1%}",
               ha="center", va="bottom", fontsize=8)

by_year.plot(kind="bar", stacked=True, ax=ax[1],
             color=["#4C78A8", "#F58518", "#E45756"])
ax[1].set_title("Class share by survey year")
ax[1].set_ylabel("share"); ax[1].tick_params(axis="x", rotation=0)
ax[1].legend(fontsize=7)
fig.tight_layout(); plt.show()

display((by_year * 100).round(2))
print("\nRows per year:"); print(df["year"].value_counts().sort_index().to_string())

The two survey years agree closely on every prevalence, which is the evidence that merging them vertically
was legitimate. The class split is roughly **84 / 2 / 14**, and `prediabetes` is rare enough that accuracy
alone will be a misleading score — the point `theory_notes.md` §4.2 makes about the original diabetes data.

---
## 3. Dataset B — Rice images

75,000 JPEGs in five class folders. What matters for the loader is that every image really is the same
size and colour mode, because the CNN legs assume a fixed tensor shape.

In [ ]:
rice_counts = {c: D._count_images(os.path.join(D.RICE_DIR, c)) for c in D.RICE_CLASSES}
print("images per class:")
for c, n in rice_counts.items():
    print(f"  {c:<12}{n:>8,}")
print(f"  {'TOTAL':<12}{sum(rice_counts.values()):>8,}")
print(f"\nperfectly balanced: {len(set(rice_counts.values())) == 1}")

# Verify the declared 250x250 RGB on a random sample rather than assuming it.
from PIL import Image
rng = np.random.default_rng(U.SEED)
sizes, modes = {}, {}
for c in D.RICE_CLASSES:
    folder = os.path.join(D.RICE_DIR, c)
    files = sorted(os.listdir(folder))
    for f in rng.choice(files, size=60, replace=False):
        with Image.open(os.path.join(folder, f)) as im:
            sizes[im.size] = sizes.get(im.size, 0) + 1
            modes[im.mode] = modes.get(im.mode, 0) + 1
print("\nsampled 300 images ->")
print("  sizes:", sizes)
print("  modes:", modes)

In [ ]:
# One example per class at native resolution, then the same grain at the 32x32
# the CNN notebooks actually train on.
fig, axes = plt.subplots(2, len(D.RICE_CLASSES), figsize=(13, 5.4))
for j, c in enumerate(D.RICE_CLASSES):
    folder = os.path.join(D.RICE_DIR, c)
    f = sorted(os.listdir(folder))[0]
    with Image.open(os.path.join(folder, f)) as im:
        im = im.convert("RGB")
        axes[0, j].imshow(im)
        axes[1, j].imshow(im.resize((32, 32), Image.BILINEAR))
    axes[0, j].set_title(c, fontsize=10)
    for i in (0, 1):
        axes[i, j].axis("off")
axes[0, 0].set_ylabel("250x250"); axes[1, 0].set_ylabel("32x32")
fig.suptitle("Rice grains: native resolution (top) and the 32x32 training input (bottom)")
fig.tight_layout(); plt.show()

**Why 32×32 is enough here.** What separates the five varieties is grain *shape*, *size* and *colour* —
Basmati is long and slender, Arborio short and round. Those survive aggressive downsampling, as the bottom
row shows. Each image is one centred grain on a black background, so there is no clutter to resolve and no
segmentation step needed. 32×32 also makes the full 75,000-image set fit comfortably in memory as float32
(~0.9 GB) and keeps the NumPy-from-scratch leg tractable, which a 250×250 input would not.

---
## 4. Dataset C — Intel scenes

This is where the inventory pays for itself.

In [ ]:
for label, base in [("seg_train", D.INTEL_TRAIN_DIR), ("seg_test", D.INTEL_TEST_DIR)]:
    root = D._intel_root(base, label)
    exists = os.path.isdir(base)
    per_class = {c: D._count_images(os.path.join(root, c)) for c in D.INTEL_CLASSES}
    print(f"{label:<10} directory exists: {str(exists):<5}  labelled images: {sum(per_class.values()):,}")

pred_root = D._intel_root(D.INTEL_PRED_DIR, "seg_pred")
pred_files = [f for f in os.listdir(pred_root) if f.lower().endswith(".jpg")]
subdirs = [d for d in os.listdir(pred_root) if os.path.isdir(os.path.join(pred_root, d))]
print(f"\nseg_pred   {len(pred_files):,} jpg files, {len(subdirs)} sub-directories")
print("-> flat file list, no class folders, therefore NO LABELS")

In [ ]:
# Confirm the guard fires with an actionable message instead of a late crash.
try:
    D.load_intel_scene()
    print("labelled splits found - notebook 03 can run")
except FileNotFoundError as e:
    print("GUARD FIRED as expected:\n")
    print(e)

In [ ]:
# seg_pred is still real imagery and is usable for an unlabelled demo.
X_pred, names = D.load_intel_unlabeled(img_size=64, n=10)
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for ax, img, nm in zip(axes, X_pred, names):
    ax.imshow(img.transpose(1, 2, 0)); ax.set_title(nm, fontsize=6); ax.axis("off")
fig.suptitle("seg_pred — real scenes, but no ground-truth class for any of them")
fig.tight_layout(); plt.show()

---
## 5. Verdict

In [ ]:
verdict = pd.DataFrame([
    ["A  BRFSS diabetes", "tabular", "546,166 rows x 19 features", "3", "YES",
     "01_diabetes_brfss.ipynb"],
    ["B  Rice images",    "image",   "75,000 x 250x250 RGB",       "5", "YES",
     "02_rice_cnn.ipynb"],
    ["C  Intel scenes",   "image",   "7,301 unlabelled only",      "6", "NO - labels missing",
     "03_intel_scene_cnn.ipynb (guarded)"],
], columns=["dataset", "kind", "usable material", "classes", "trainable?", "notebook"])
display(verdict)

**Two datasets can be trained on, one cannot.**

- **BRFSS diabetes** and **Rice** are complete and go straight into notebooks 01 and 02. Between them they
  cover both halves of the assignment's argument: a tabular problem where convolution has nothing to
  exploit, and an image problem where it does.
- **Intel scenes** is written up in notebook 03 and will run unchanged the moment `seg_train`/`seg_test`
  are restored with
  `kaggle datasets download -d puneet6060/intel-image-classification -p data/ --unzip`.
  Until then it contributes only an inference demo, and no number from it appears in any results table.

Notebook 04 then runs the M1→M4 improvement ladder from `theory_notes.md` §3, and notebook 05 collects
everything into the cross-dataset comparison.